# Sprint: APIs REST con Python, requests y pandas

Contenido:
- **Nivel 1**: JSONPlaceholder (GET, POST, PATCH, DELETE y códigos de estado)
- **Nivel 2**: API pública real (PokéAPI) → DataFrame
- **Nivel 3**: Open Data BCN (CKAN) → DataFrame → CSV

In [ ]:
import requests
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

# Nivel 1 · JSONPlaceholder

In [ ]:
BASE_URL = "https://jsonplaceholder.typicode.com"

## 1 y 2. GET de `/posts`, `/users` y `/todos` + cantidad y código de estado

In [ ]:
recursos = ["posts", "users", "todos"]
datos = {}

for recurso in recursos:
    resp = requests.get(f"{BASE_URL}/{recurso}", timeout=10)
    datos[recurso] = resp.json()
    print(f"/{recurso:<6} -> código de estado: {resp.status_code} | total de registros: {len(datos[recurso])}")

## 3. Petición a una publicación inexistente (404)

In [ ]:
resp = requests.get(f"{BASE_URL}/posts/99999", timeout=10)
print("Código de estado:", resp.status_code)
print("Cuerpo de la respuesta:", resp.json())
print("¿Respuesta correcta (ok)?", resp.ok)

## 4. POST: crear una publicación ficticia

In [ ]:
nueva_publicacion = {
    "title": "Mi primera publicación",
    "body": "Este es el contenido de una publicación creada con requests.",
    "userId": 1,
}

resp = requests.post(f"{BASE_URL}/posts", json=nueva_publicacion, timeout=10)
print("Código de estado:", resp.status_code)
print("Respuesta JSON:")
print(resp.json())

> JSONPlaceholder **simula** la creación: devuelve `201 Created` y un `id` nuevo (101), pero no guarda nada realmente.

## 5. PATCH: modificar parcialmente una publicación

In [ ]:
cambios = {"title": "Título modificado con PATCH"}

resp = requests.patch(f"{BASE_URL}/posts/1", json=cambios, timeout=10)
print("Código de estado:", resp.status_code)
print("Respuesta JSON:")
print(resp.json())

> Con PATCH solo se envían los campos a cambiar; el resto de campos (`body`, `userId`) se mantienen.

## 6. DELETE sobre una publicación

In [ ]:
resp = requests.delete(f"{BASE_URL}/posts/1", timeout=10)
print("Código de estado:", resp.status_code)
print("Respuesta JSON:", resp.json())

> El DELETE devuelve `200` y un JSON vacío `{}`.

# Nivel 2 · API pública real: PokéAPI

He elegido **[PokéAPI](https://pokeapi.co/)** del repositorio [public-apis](https://github.com/public-apis/public-apis) (categoría *Games & Comics*): no requiere autenticación, admite GET y responde en **JSON**.

## Documentación

### Endpoints
| Endpoint | Descripción |
|---|---|
| `GET /api/v2/pokemon` | Lista paginada de Pokémon |
| `GET /api/v2/pokemon/{id o nombre}` | Detalle de un Pokémon (altura, peso, tipos, habilidades...) |
| `GET /api/v2/type` | Lista de tipos (fuego, agua, planta...) |
| `GET /api/v2/ability` | Lista de habilidades |

### Parámetros opcionales
- `limit`: número de resultados por página (por defecto 20).
- `offset`: posición desde la que empezar (paginación).

### Formato de respuesta
JSON. Las listas llegan con la estructura `{count, next, previous, results: [...]}`.

## 3. Petición GET sencilla

In [ ]:
POKE_URL = "https://pokeapi.co/api/v2"

resp = requests.get(f"{POKE_URL}/pokemon", params={"limit": 20, "offset": 0}, timeout=10)
print("Código de estado:", resp.status_code)
print("Content-Type:", resp.headers.get("Content-Type"))

data = resp.json()
print("\nTotal de Pokémon en la API:", data["count"])
print("Siguiente página:", data["next"])
print("\nPrimeros 5 resultados:")
for p in data["results"][:5]:
    print(f"  - {p['name']:<12} {p['url']}")

Un segundo endpoint, el detalle de un Pokémon:

In [ ]:
resp = requests.get(f"{POKE_URL}/pokemon/pikachu", timeout=10)
pikachu = resp.json()
print("Código de estado:", resp.status_code)
print("Nombre:", pikachu["name"])
print("Altura:", pikachu["height"], "| Peso:", pikachu["weight"])
print("Tipos:", [t["type"]["name"] for t in pikachu["types"]])
print("Habilidades:", [a["ability"]["name"] for a in pikachu["abilities"]])

## 4. Conversión a DataFrame

In [ ]:
# Lista básica -> DataFrame
df_lista = pd.DataFrame(data["results"])
df_lista.head()

In [ ]:
# Enriquecemos con el detalle de cada Pokémon (altura, peso, experiencia base, tipos)
filas = []
for p in data["results"]:
    d = requests.get(p["url"], timeout=10).json()
    filas.append({
        "id": d["id"],
        "nombre": d["name"],
        "altura": d["height"],
        "peso": d["weight"],
        "experiencia_base": d["base_experience"],
        "tipos": ", ".join(t["type"]["name"] for t in d["types"]),
    })

df_pokemon = pd.DataFrame(filas)
df_pokemon.head()

# Nivel 3 · Open Data BCN

La API de Open Data BCN está basada en **CKAN**. URL base: `https://opendata-ajuntament.barcelona.cat/data/api/3/action/`

## 1. Buscar un dataset (`package_search` y `package_show`)

In [ ]:
CKAN_URL = "https://opendata-ajuntament.barcelona.cat/data/api/3/action"

# Cambia el término de búsqueda por el tema que te interese
QUERY = "bicing"

resp = requests.get(f"{CKAN_URL}/package_search", params={"q": QUERY, "rows": 10}, timeout=20)
print("Código de estado:", resp.status_code)

resultado = resp.json()["result"]
print(f"Datasets encontrados: {resultado['count']}\n")

for pkg in resultado["results"]:
    formatos = sorted({r["format"] for r in pkg["resources"]})
    print(f"- {pkg['name']:<45} formatos: {formatos}")

## 2. Seleccionar un dataset con recursos CSV o JSON (y datastore activo)

In [ ]:
dataset_elegido, recurso_elegido = None, None

for pkg in resultado["results"]:
    for r in pkg["resources"]:
        if r["format"].upper() in ("CSV", "JSON") and r.get("datastore_active"):
            dataset_elegido, recurso_elegido = pkg["name"], r
            break
    if recurso_elegido:
        break

if recurso_elegido is None:
    raise ValueError("Ningún resultado tiene un recurso CSV/JSON con datastore activo. Prueba otra búsqueda (QUERY).")

# package_show: detalle del dataset elegido
resp = requests.get(f"{CKAN_URL}/package_show", params={"id": dataset_elegido}, timeout=20)
detalle = resp.json()["result"]

print("Dataset:", detalle["title"])
print("Recursos disponibles:")
for r in detalle["resources"]:
    print(f"  - {r['name'][:50]:<50} {r['format']:<5} datastore_active={r.get('datastore_active')}  id={r['id']}")

resource_id = recurso_elegido["id"]
print("\nRecurso seleccionado:", recurso_elegido["name"], "->", resource_id)

## 3. `datastore_search`: recuperar al menos 100 registros

In [ ]:
resp = requests.get(
    f"{CKAN_URL}/datastore_search",
    params={"resource_id": resource_id, "limit": 100},
    timeout=30,
)
print("Código de estado:", resp.status_code)

res = resp.json()["result"]
registros = res["records"]
print("Registros totales en el recurso:", res.get("total"))
print("Registros recuperados:", len(registros))

## 4. Conversión a DataFrame

In [ ]:
df_bcn = pd.DataFrame(registros)
print(df_bcn.shape)
df_bcn.head()

## 5. Guardar el DataFrame en un archivo `.csv`

In [ ]:
df_bcn.to_csv("opendata_bcn.csv", index=False, encoding="utf-8")
print("Archivo guardado: opendata_bcn.csv")

# Comprobación
pd.read_csv("opendata_bcn.csv").head()